In [ ]:
import degirum as dg
import degirum_tools
from picamera2 import Picamera2
import cv2
import numpy as np
import pandas

In [2]:
model = dg.load_model(
    model_name="depth_anything_v2--224x224_quant_hailort_hailo8_1",
    inference_host_address="@local",
    zoo_url="/home/rohan/degirum_wearable_device/models"
)

In [3]:
camera = Picamera2()
camera.configure(camera.create_video_configuration(main={"format": "RGB888"}))
camera.start()

[0:19:15.914584696] [4367]  INFO Camera camera_manager.cpp:326 libcamera v0.5.0+59-d83ff0a4
[0:19:15.925539556] [4872]  INFO RPI pisp.cpp:720 libpisp version v1.2.1 981977ff21f3 29-04-2025 (14:13:50)
[0:19:15.942280716] [4872]  INFO RPI pisp.cpp:1179 Registered camera /base/axi/pcie@1000120000/rp1/i2c@88000/imx500@1a to CFE device /dev/media0 and ISP device /dev/media2 using PiSP variant BCM2712_D0
[0:19:15.949180776] [4367]  INFO Camera camera.cpp:1205 configuring streams: (0) 1280x720-RGB888 (1) 2028x1520-RGGB_PISP_COMP1
[0:19:15.949318313] [4872]  INFO RPI pisp.cpp:1483 Sensor: /base/axi/pcie@1000120000/rp1/i2c@88000/imx500@1a - Selected sensor format: 2028x1520-SRGGB10_1X10 - Selected CFE format: 2028x1520-PC1R


In [4]:
with degirum_tools.Display("Depth Camera") as output_display:
    while True:
        frame = camera.capture_array()
        result = model.predict(frame)

        # Extract the raw data
        raw_data = result._inference_results[0]['data']

        average_depth = np.mean(raw_data)
        print(f"Average Depth: {average_depth}")

        # Dequantize the data
        scale = result._inference_results[0]['quantization']['scale'][0]
        zero = result._inference_results[0]['quantization']['zero'][0]
        depth_map = (raw_data.astype(np.float32) - zero) * scale

        # Reshape to 2D (224x224)
        depth_map = depth_map.reshape((224, 224))

        # Normalize to 0-255 for visualization
        depth_map_normalized = cv2.normalize(depth_map, None, 0, 255, cv2.NORM_MINMAX)
        depth_map_normalized = np.uint8(depth_map_normalized)

        depth_map_color = cv2.cvtColor(depth_map_normalized, cv2.COLOR_GRAY2BGR)

        output_display.show(depth_map_color)

Average Depth: 40.953204719387756
Average Depth: 41.3962252869898
Average Depth: 39.698521205357146
Average Depth: 39.039142219387756
Average Depth: 35.17233338647959
Average Depth: 39.347676179846935
Average Depth: 41.49141023596939
Average Depth: 34.1611328125
Average Depth: 32.66206951530612
Average Depth: 37.27118542729592
Average Depth: 30.9232102997449
Average Depth: 31.851662149234695
Average Depth: 38.29791135204081
Average Depth: 39.05668048469388
Average Depth: 39.40543287627551
Average Depth: 42.941286670918366
Average Depth: 37.7556002869898
Average Depth: 34.29286910076531
Average Depth: 29.452268016581634
Average Depth: 40.84229512117347
Average Depth: 28.11493542729592
Average Depth: 34.898995535714285
Average Depth: 37.76000478316327
Average Depth: 41.73670679209184
Average Depth: 32.04848931760204
Average Depth: 37.35269850127551
Average Depth: 36.02266023596939
Average Depth: 38.10259885204081
Average Depth: 38.81407445790816
Average Depth: 37.10849808673469
Average D

In [5]:
import time

# Initialize min and max
min_depth = float('inf')
max_depth = float('-inf')
start_time = time.time()
duration = 30  # seconds

print("Collecting min/max average_depth for 30 seconds...")
while time.time() - start_time < duration:
    frame = camera.capture_array()
    result = model.predict(frame)
    raw_data = result._inference_results[0]['data']
    average_depth = np.mean(raw_data)
    min_depth = min(min_depth, average_depth)
    max_depth = max(max_depth, average_depth)
    print(f"Current: {average_depth:.2f}, Min: {min_depth:.2f}, Max: {max_depth:.2f}")
    time.sleep(0.1)  # Adjust as needed

print(f"Final min: {min_depth:.2f}, max: {max_depth:.2f}")

# Example: Map a new average_depth to vibrator value (20 to 127)
def map_depth_to_vibrator(depth, min_d, max_d, min_v=20, max_v=127):
    # Avoid division by zero
    if max_d == min_d:
        return min_v
    # Linear mapping
    return int(min_v + (depth - min_d) * (max_v - min_v) / (max_d - min_d))

# Example usage:
new_average_depth = np.mean(raw_data)  # Replace with new value as needed
vibrator_value = map_depth_to_vibrator(new_average_depth, min_depth, max_depth)
print(f"Mapped vibrator value: {vibrator_value}")



Current: 40.21, Min: 40.21, Max: 40.21
Current: 41.91, Min: 40.21, Max: 41.91
Current: 42.41, Min: 40.21, Max: 42.41
Current: 44.12, Min: 40.21, Max: 44.12
Current: 46.16, Min: 40.21, Max: 46.16
Current: 48.82, Min: 40.21, Max: 48.82
Current: 45.37, Min: 40.21, Max: 48.82
Current: 35.16, Min: 35.16, Max: 48.82
Current: 41.34, Min: 35.16, Max: 48.82
Current: 44.56, Min: 35.16, Max: 48.82
Current: 36.78, Min: 35.16, Max: 48.82
Current: 36.79, Min: 35.16, Max: 48.82
Current: 51.18, Min: 35.16, Max: 51.18
Current: 56.11, Min: 35.16, Max: 56.11
Current: 45.69, Min: 35.16, Max: 56.11
Current: 34.08, Min: 34.08, Max: 56.11
Current: 39.39, Min: 34.08, Max: 56.11
Current: 28.53, Min: 28.53, Max: 56.11
Current: 37.31, Min: 28.53, Max: 56.11
Current: 32.50, Min: 28.53, Max: 56.11
Current: 28.24, Min: 28.24, Max: 56.11
Current: 31.39, Min: 28.24, Max: 56.11
Current: 36.66, Min: 28.24, Max: 56.11
Current: 33.86, Min: 28.24, Max: 56.11
Current: 31.81, Min: 28.24, Max: 56.11
Current: 40.31, Min: 28.2